# 6.2 Code Brief: Neural Networks

Condensed reference for notebook 6.2.

## Neural Network Basics
- **Input layer**: One neuron per feature
- **Hidden layers**: Neurons that transform inputs through weighted sums and activation functions
- **Output layer**: Produces the final prediction

Each neuron computes: $output = activation(\sum(weights \times inputs) + bias)$

### Common Activation Functions
- **ReLU**: $f(x) = max(0, x)$ — most common for hidden layers
- **Sigmoid**: $f(x) = 1/(1+e^{-x})$ — used for binary classification output
- **Softmax**: Used for multi-class output

## Setup and Data Preparation

In [ ]:
# Build a neural network using scikit-learn
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

from google.colab import drive
drive.mount('/content/drive')

# Load data (same preparation as Module 2)
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
train_df = pd.read_csv(f'{filepath}training.csv')
test_df = pd.read_csv(f'{filepath}testing.csv')

train_df['DEPARTED'] = (train_df['SEM_3_STATUS'] != 'E').astype(int)
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

train_enc = pd.get_dummies(train_df[numeric_features + categorical_features],
                           columns=categorical_features, drop_first=True)
test_enc = pd.get_dummies(test_df[numeric_features + categorical_features],
                          columns=categorical_features, drop_first=True)
train_enc, test_enc = train_enc.align(test_enc, join='left', axis=1, fill_value=0)

# Impute with TRAIN medians only, never test's own (avoids leakage)
train_medians = train_enc.median()
train_enc = train_enc.fillna(train_medians)
test_enc = test_enc.fillna(train_medians)

X_train, y_train = train_enc, train_df['DEPARTED']
X_test, y_test = test_enc, test_df['DEPARTED']

# IMPORTANT: Neural networks REQUIRE feature scaling (unlike tree-based models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Build the Neural Network

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Build neural network — still follows instantiate/fit/predict!
nn = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),  # Three hidden layers. These values are largely your decision
    activation='relu',
    solver='adam',
    alpha=0.001,  # L2 regularization
    batch_size=50,
    learning_rate='adaptive',
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=88
)

nn.fit(X_train_scaled, y_train)
nn_prob = nn.predict_proba(X_test_scaled)[:, 1]

print(f"Neural Network ROC-AUC: {roc_auc_score(y_test, nn_prob):.4f}")
print(f"\nNote: Neural networks require scaled features!")
print(f"Note: They also take longer to train and have more hyperparameters.")

## Neural Networks vs. Tree-Based Models for Tabular Data

| Aspect | Neural Networks | Tree-Based Models |
|:-------|:---------------|:-----------------|
| **Preprocessing** | Requires scaling | None needed |
| **Interpretability** | Black box | Moderate to high |
| **Tuning effort** | High (many hyperparameters) | Moderate |
| **Performance on tabular data** | Good, sometimes great | Typically best |
| **Training speed** | Slow | Moderate to fast |
| **Sample efficiency** | Needs more data | Works with less |

## Key Takeaways
- Neural networks follow the same scikit-learn pattern: `MLPClassifier().fit().predict()`
- They **require feature scaling** (unlike tree models)
- For tabular student data, they rarely outperform Random Forest or XGBoost
- Best suited for non-tabular data (images, text, sequences)

**Next:** Module 7 — EDA in Unstructured Data